# Hidden Markov Models (HMMs)

_Artificial Intelligence (CS221)_

**You can't see the true state, only noisy clues. Infer the hidden truth over time.**

Some things you cannot observe directly. You only see clues.

     A Hidden Markov Model has a hidden state that changes over time, and observations that hint at it.

---

This notebook builds the HMM **forward algorithm** (filtering) step by step. Each day we *predict* where the hidden state probably drifted, then *update* that guess using what we observed, then *renormalize* into a clean probability. Run each cell, read the note above it, then watch the belief "it is raining" shift across several days. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

The classic "umbrella world": the hidden state is whether it's **raining**, and each day we only observe whether someone carried an **umbrella**. We build the filter in three steps: (1) define the transition, emission, and prior, (2) run one predict-update-renormalize step, (3) loop it over several days of observations.

### Step 1 — Define the model: transitions, emissions, prior

An HMM needs three pieces:
- **Transition matrix `T`** — how the hidden state evolves. `T[i, j]` is the probability of moving from state `i` today to state `j` tomorrow. Here weather is "sticky": rain tends to follow rain.
- **Emission `E_umb`** — how the hidden state produces an observation. `E_umb[s]` is the probability of *seeing an umbrella* given state `s`. Rain almost always means an umbrella; sun rarely does.
- **Prior `belief`** — our initial guess before any observation, 50/50.

In [ ]:
# Hidden states: index 0 = Rain, index 1 = No rain.
T = np.array([
    [0.7, 0.3],   # P(next state | today = Rain)
    [0.3, 0.7],   # P(next state | today = No rain)
])

# Emission: P(umbrella seen | state). Rain -> usually umbrella; sun -> rarely.
E_umb = np.array([0.9, 0.2])

# Prior belief over states before seeing anything: 50/50.
prior = np.array([0.5, 0.5])

print("transition T:\n", T)
print("emission P(umbrella | state):", E_umb)
print("prior belief:", prior)

### Step 2 — One filtering step: predict, update, renormalize

The forward algorithm advances the belief one day at a time in three moves:
1. **Predict** — push the current belief through the transition matrix (`belief @ T`) to account for the state possibly changing overnight.
2. **Update** — multiply by the likelihood of what we observed. If we saw an umbrella, weight by `E_umb`; if not, weight by `1 - E_umb`.
3. **Renormalize** — divide by the total so the belief sums back to `1` and stays a valid probability distribution.

Let's run a single day where we *did* see an umbrella.

In [ ]:
belief = prior
saw_umbrella = True

# 1. Predict: push belief through the transitions.
belief = belief @ T

# 2. Update: weight by the likelihood of this observation.
likelihood = E_umb if saw_umbrella else (1 - E_umb)
belief = belief * likelihood

# 3. Renormalize back into a probability distribution.
belief = belief / belief.sum()

print("after day 0 (umbrella=True):  P(rain) = %.3f" % belief[0])

### Step 3 — Loop the filter over a sequence of days

Now we chain the same predict-update-renormalize step across a list of daily observations. Each day's belief becomes the starting point for the next. Watch how two umbrella days push `P(rain)` up, and a no-umbrella day pulls it back down.

In [ ]:
observations = [True, True, False]   # umbrella, umbrella, no umbrella
belief = prior

for day, saw_umbrella in enumerate(observations):
    belief = belief @ T                           # predict: push through transitions

    likelihood = E_umb if saw_umbrella else (1 - E_umb)
    belief = belief * likelihood                  # update: weight by the observation

    belief = belief / belief.sum()                # renormalize

    print("day", day,
          "umbrella=%-5s" % saw_umbrella,
          "P(rain) = %.3f" % belief[0])

## Visualize the data & results

_A guard never sees outside but watches whether the director brings an umbrella — how does the belief 'it is raining' shift over five days?_

We'll run the same filter over five days of observations and plot `P(rain)` as it evolves.

### Step 1 — Filter five days and record P(rain) each day

Same model and same predict-update-renormalize loop as before, but over five observations. This time we append `belief[0]` — the probability of rain — to a list so we can plot the whole trajectory.

In [ ]:
T = np.array([[0.7, 0.3], [0.3, 0.7]])   # rain/sun -> rain/sun
E_umb = np.array([0.9, 0.2])             # P(umbrella | rain), P(umbrella | sun)
belief = np.array([0.5, 0.5])

observations = [True, True, False, True, True]
p_rain = []

for saw_umbrella in observations:
    belief = belief @ T                           # predict

    likelihood = E_umb if saw_umbrella else (1 - E_umb)
    belief = belief * likelihood                  # update by observation

    belief = belief / belief.sum()                # renormalize
    p_rain.append(belief[0])

print("P(rain) per day:", [round(p, 3) for p in p_rain])

### Step 2 — Plot the filtered belief over time

The line shows how the guard's belief that "it is raining" rises on umbrella days and dips on the no-umbrella day. The belief is anchored between 0 and 1 because it's a probability.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, 6), p_rain, "-o", color="#4ea1ff")

ax.set_xticks(range(1, 6))
ax.set_xlabel("day")
ax.set_ylabel("P(rain)")
ax.set_ylim(0, 1)
ax.set_title("umbrella world: filtered belief P(rain) over 5 days")
plt.show()